In [1]:
!pip -q install transformers accelerate evaluate scikit-learn

In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

In [3]:
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [4]:
# Load the training data used in the project
with open("records_long.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(df.head())
print(df.columns.tolist())
print(df.shape)

     model                                               text      topic  \
0    human  In Minnesota and North Dakota , they tend to u...  chemistry   
1    human  Used to work in the salt industry ( yes , it '...  chemistry   
2    human  Its crystalline structure can also be manipula...  chemistry   
3  chatgpt  Salt is used on roads to help melt ice and sno...  chemistry   
4  chatgpt  However, salt is often the most effective and ...  chemistry   

  origin  length  
0    HC3     550  
1    HC3     456  
2    HC3     581  
3    HC3     547  
4    HC3     598  
['model', 'text', 'topic', 'origin', 'length']
(320593, 5)


In [5]:
df = df[["text", "model"]].copy()
df = df.dropna(subset=["text", "model"])
df["text"] = df["text"].astype(str)

print(df.shape)
print(df["model"].value_counts())

(320593, 2)
model
llama      86120
mistral    81448
gemma      69388
human      49050
chatgpt    34587
Name: count, dtype: int64


In [6]:
# Keep the same label mapping used in your final notebook
label_mapping = {
    "chatgpt": "OpenAI",
    "gemma": "Google",
    "human": "Human",
    "llama": "Meta",
    "mistral": "Anthropic"
}

df["final_label"] = df["model"].map(label_mapping)
df = df.dropna(subset=["final_label"])

print(df["final_label"].value_counts())
print(df.shape)

final_label
Meta         86120
Anthropic    81448
Google       69388
Human        49050
OpenAI       34587
Name: count, dtype: int64
(320593, 3)


In [7]:
# Better choice: use the full dataset
# If training is too slow, sample ONLY ONCE, not twice.

USE_SAMPLE = False
SAMPLE_FRAC = 0.7   # only used if USE_SAMPLE = True

if USE_SAMPLE:
    df = (
        df.groupby("model", group_keys=False)
          .apply(lambda x: x.sample(frac=SAMPLE_FRAC, random_state=42))
          .reset_index(drop=True)
    )

print(df.shape)
print(df["final_label"].value_counts())

(320593, 3)
final_label
Meta         86120
Anthropic    81448
Google       69388
Human        49050
OpenAI       34587
Name: count, dtype: int64


In [8]:
encoder = LabelEncoder()
df["label_id"] = encoder.fit_transform(df["final_label"])

print("Class mapping:")
for i, class_name in enumerate(encoder.classes_):
    print(f"{i} -> {class_name}")

Class mapping:
0 -> Anthropic
1 -> Google
2 -> Human
3 -> Meta
4 -> OpenAI


In [9]:
train_texts, val_texts, y_train, y_val = train_test_split(
    df["text"].tolist(),
    df["label_id"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))

Training samples: 256474
Validation samples: 64119


In [10]:
# Fast and safe on Colab:
MODEL_NAME = "distilbert-base-uncased"

# Alternatives:
# MODEL_NAME = "bert-base-uncased"   # classic BERT
# MODEL_NAME = "roberta-base"        # often strong, but a bit heavier
# MODEL_NAME = "distilroberta-base"  # nice compromise

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

C:\Users\franc\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\franc\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False
        )

        item = {key: torch.tensor(val) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [13]:
MAX_LENGTH = 256   # try 384 later if texts are long and GPU is ok

train_dataset = TextDataset(train_texts, y_train, tokenizer, max_length=MAX_LENGTH)
val_dataset = TextDataset(val_texts, y_val, tokenizer, max_length=MAX_LENGTH)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")

    return {
        "accuracy": acc,
        "f1_macro": f1
    }

In [15]:
id2label = {i: label for i, label in enumerate(encoder.classes_)}
label2id = {label: i for i, label in enumerate(encoder.classes_)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(encoder.classes_),
    id2label=id2label,
    label2id=label2id
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
training_args = TrainingArguments(
    output_dir="results/bert_run",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [18]:
trainer.train()

C:\Users\franc\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

In [ ]:
# Load the professor file / submission file
df_subm = pd.read_csv("dataset-exemplos.csv", sep=";")

print("\nSubmission file loaded.")
print(df_subm.head())
print("\nSubmission columns:", df_subm.columns.tolist())
print("Submission shape:", df_subm.shape)

In [ ]:
# Detect the text column automatically
possible_text_columns = ["text", "Text", "sentence", "Sentence", "content", "Content"]

text_column = None
for col in possible_text_columns:
    if col in df_subm.columns:
        text_column = col
        break

if text_column is None:
    raise ValueError(
        "Could not find a text column in dataset-exemplos.csv. "
        "Rename the correct column or edit 'possible_text_columns'."
    )

print(f"\nUsing text column for prediction: {text_column}")

df_subm[text_column] = df_subm[text_column].astype(str)
df_subm["clean_text"] = df_subm[text_column].apply(clean_text)

In [ ]:
# Encode the professor texts using the same vocabulary
X_subm = [encode_text(text, word_index, max_len=max_len) for text in df_subm["clean_text"]]

In [ ]:
predictions = []

model.eval()
with torch.no_grad():
    for start in range(0, len(X_subm), 64):
        end = start + 64

        batch = torch.tensor(X_subm[start:end], dtype=torch.long).to(device)
        outputs = model(batch)
        _, predicted = torch.max(outputs, 1)

        predictions.extend(predicted.cpu().numpy())

predicted_labels = encoder.inverse_transform(predictions)

In [ ]:
# Add predictions to the final dataframe
df_subm["Predicted_Label"] = predicted_labels

print("\nSample predictions:")
print(df_subm[[text_column, "Predicted_Label"]].head())

# Save a CSV that keeps the original text and the predicted label
output_columns = [text_column, "Predicted_Label"]
if "Label" in df_subm.columns:
    output_columns.append("Label")

df_subm[output_columns].to_csv("results/submission_lstm.csv", index=False)
print("\nSaved file: results/submission_lstm.csv")